# Replicate draws on Colab — GRPO_LA5 @10 and PTO_LA0 @10

Runs the two **replicate** generate-only passes (STATUS.md § replicate draw) on an **A100
runtime** by driving `tools/generate_eval_convs.py --method {grpo,pto}` — Run All is enough.

- Uses the stored A100 `conversation_batch_size` (no `--batch-size` override), so each 96-conv
  pass takes roughly 15–30 min. The two run sequentially in this one notebook.
- Output goes to `conversations/replicate/<EXP>/model_iter_10_rep1_TT0.9_TP0.7/` — **outside**
  `conversations/full/`, so auto-discovery never sees it (deliberate; the lake names carry a
  `_rep1_` infix instead). Resume-safe per conversation CSV: rerunning skips what exists.
- Keys come from Colab Secrets (`OPENAI_API_KEY`, `huggingface`), as in the trainer notebooks.
- The `seed + k + 1` shuffle convention was verified against both arms with `--verify-seeds`
  on 2026-08-26 (726/717 correct, 0 wrong, decoy offsets fail), so this notebook skips it.
- Afterwards: let Drive Desktop sync (tray ✓), then score locally with
  `eda/tools/score_replicate.py --primary --judge` and analyse with `eda/tools/replicate_check.py`.

In [ ]:
# Colab pre-bakes torchao < 0.16.0, which peft 0.19.1 rejects inside PeftModel.from_pretrained
# (dispatch_torchao) — this pass loads an adapter, so it hits it too. The project never uses
# torchao. No-op off Colab.
%pip uninstall -y -q torchao

# Uncomment on a FRESH runtime if imports fail (pins from requirements.txt; torch stays Colab's):
# %pip install -q peft==0.19.1 transformers==5.8.1 trl==1.4.1 accelerate==1.13.0 openai==2.36.0

In [ ]:
# ── Draw 1: GRPO K=5, iteration 10 (the best final state — and the one with the worst
# per-conversation cross-judge agreement, which is why it gets a replicate) ──
%run /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/tools/generate_eval_convs.py --method grpo --iter 10 --experiment GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8 --conv-dir /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/grpo_Exp3/conversations/replicate/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/model_iter_10_rep1_TT0.9_TP0.7

In [ ]:
# ── Draw 2: PTO K=0, iteration 10 (the arm GRPO K=5 beats by 0.007 on the held-out judge) ──
%run /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/tools/generate_eval_convs.py --method pto --iter 10 --experiment PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy --conv-dir /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/pto_Exp3/conversations/replicate/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/model_iter_10_rep1_TT0.9_TP0.7

In [ ]:
import os
for d in [r"/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/grpo_Exp3/conversations/replicate/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/model_iter_10_rep1_TT0.9_TP0.7", r"/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/pto_Exp3/conversations/replicate/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/model_iter_10_rep1_TT0.9_TP0.7"]:
    n = len([f for f in os.listdir(d) if f.startswith("conversation_")]) if os.path.isdir(d) else 0
    print(("OK  " if n >= 96 else "SHORT"), n, "/96 —", d.split("/data/")[-1])
print("Both at 96/96 -> let Drive Desktop finish syncing, then run score_replicate.py locally.")